# Job Market Intelligence - Data Cleaning & Preprocessing Pipeline
This notebook documents the data extraction, normalization, skill tokenization, and schema structuring workflow for the Job Market Intelligence & Skill Demand Analytics platform.

In [ ]:
import os
import sys
import django
import pandas as pd
import numpy as np

# Setup Django environment
sys.path.append('..')
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'job_market.settings')
django.setup()

from jobs.models import Job, Company, Skill
print('Django Environment initialized successfully!')

## 1. Load Raw Job Data into Pandas DataFrame

In [ ]:
jobs_qs = Job.objects.select_related('company').prefetch_related('skills').all()
data = []
for j in jobs_qs:
    skills = [s.name for s in j.skills.all()]
    data.append({
        'id': j.id,
        'title': j.title,
        'company': j.company.name,
        'industry': j.company.industry,
        'role_category': j.role_category,
        'location': j.location,
        'experience_level': j.experience_level,
        'experience_min': j.experience_min,
        'experience_max': j.experience_max,
        'work_mode': j.work_mode,
        'salary_min': j.salary_min,
        'salary_max': j.salary_max,
        'salary_avg': j.salary_avg,
        'skills_count': len(skills),
        'skills': ', '.join(skills),
        'posted_date': j.posted_date,
    })

df = pd.DataFrame(data)
print(f'Total records loaded: {len(df)}')
df.head()

## 2. Check Missing Values & Duplicates

In [ ]:
print('Missing Values:\n', df.isnull().sum())
print('\nDuplicate rows:', df.duplicated(subset=['title', 'company', 'location']).sum())

## 3. Skill Normalization & Frequency Distribution

In [ ]:
skill_counts = df['skills'].str.split(', ').explode().value_counts()
print('Top 15 Most Demanded Skills Overall:')
print(skill_counts.head(15))